# Session 18 — Project Refactor Day (Workshop Notebook)

This notebook is the **hands-on refactor path** for Module 6, Session 18.

**Goal:** start from a prototype travel planner and refactor it to be robust using:
- **Retries** (bounded attempts + backoff)
- **Exception guards** (safe calls + validation)
- **Failure routing** in a **LangGraph state machine**

> Tip for trainers: run the baseline notebook first (`baseline/travel_planner.ipynb`), trigger a couple failures, then follow this notebook step-by-step for the refactor.


## 0) Setup

From the project root:

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
source .venv/bin/activate
pip install -r requirements.txt

# Optional but recommended (no paid APIs):
# Install Ollama and pull a small model, e.g.:
# ollama pull llama3.1:8b
```

Then set env vars:
- copy `.env.example` → `.env`
- keep `PROVIDER=ollama` for free local runs


In [ ]:
import os
from pathlib import Path

print('CWD:', Path.cwd())
print('PROVIDER:', os.getenv('PROVIDER', 'not set'))


## 1) What we are refactoring

**BEFORE:** a prototype travel planner tends to assume happy paths.

**AFTER:** we enforce reliability with a repeatable pattern:
1. Define a **typed state**
2. Wrap risky calls with **guards**
3. Add **retry policy** for transient errors
4. Route failures to a **fallback node** (no crashes)

We’ll implement these using small reusable utilities, then plug them into a LangGraph `StateGraph`.


## 2) Import the refactor utilities (guards + retries)

These are the same patterns used in the practical kit’s `src/` modules.


In [ ]:
from src.retry import RetryPolicy, retry
from src.guards import SafeCallError, safe_call

RetryPolicy()

## 3) Define the Travel Planner state

A typed state makes the graph predictable:
- each node reads/writes known fields
- guards can validate required fields


In [ ]:
from src.state import TravelState

# TravelState is a TypedDict used by LangGraph nodes
TravelState.__annotations__

## 4) Tools and LLM wrapper

To keep this class mostly offline, tools are **mocked** but realistic:
- They can simulate transient failures (to teach retries)
- They return structured data the planner can use

The LLM uses **Ollama by default** (no paid API). You can switch providers later.


In [ ]:
from src.tools import search_flights, search_hotels, suggest_activities
from src.llm import get_llm

llm = get_llm()
llm

## 5) The refactored LangGraph

This graph contains:
- Nodes: `collect -> research -> plan -> critique -> revise`
- A **loop** (state machine) from critique back to revise when quality is low
- **Failure routing** to a `fallback` node if a node raises or returns invalid output

Open `src/graph.py` and `src/nodes.py` after this cell to see the structure.


In [ ]:
from src.graph import build_graph

graph = build_graph()
print(graph)

## 6) Run scenarios (this is your “before vs after” proof)

We’ll run three scenarios:
1) Clean input (happy path)
2) Missing fields (guard should stop and ask)
3) Transient tool failures (retry should recover)


In [ ]:
from src.main import run_once

# 1) Clean request
result1 = run_once(
    user_request="Plan a 4-day budget trip to Jaipur for 2 people in March. Include food and culture.",
    simulate_failures=False,
)
print("\n--- RESULT 1 ---\n", result1)


In [ ]:
# 2) Missing fields (guards should respond gracefully)
result2 = run_once(
    user_request="Plan a trip.",
    simulate_failures=False,
)
print("\n--- RESULT 2 ---\n", result2)


In [ ]:
# 3) Transient failures: tools randomly fail, retries should kick in
result3 = run_once(
    user_request="Plan a 3-day trip to Goa for 2 people under INR 25k with beaches and local food.",
    simulate_failures=True,
)
print("\n--- RESULT 3 ---\n", result3)


## 7) Trainer notes: what to emphasize

- **Retry only transient errors** (timeouts, temporary API failure)
- **Guards catch permanent errors** (missing fields, schema mismatch)
- Prefer **fallback outputs** over crashes (partial results are better than none)
- In LangGraph, robustness is *often just another edge*: `on_error -> fallback`

Next, we can do the *guided in-class refactor* directly on the baseline notebook by copying these patterns step-by-step.
